# TP2 – Representación vectorial de texto: embeddings y búsqueda semántica
**UNR · TUIA · Procesamiento de Lenguaje Natural – Unidad 2**

**Integrantes:**

**Librerias utilizadas**

In [1]:
#!pip install nltk
#!pip install psycopg2-binary

import json
import re
import nltk
import numpy as np
import pandas as pd
import getpass
import os
import psycopg2

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [2]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Carolina\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Carolina\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Carolina\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

**Parte A - Preparación del corpus**

*Extracción del archivo csv del TP parte 1*

In [3]:
#Ruta del csv extraido de la parte 1 del TP
archivo_csv = "../data/libros.csv"

#Lectura del csv con pandas
df_libros = pd.read_csv(archivo_csv, dtype=str, keep_default_na=False)
#dtype str fuerza a todos los campos a ser leidos como string,
#keep_default_na=False evita que se reemplacen los valores vacios por NaN

#print(df_libros.head())

Se realiza una copia del archivo csv para no modificar el original

In [4]:
df_sinopsis = df_libros[df_libros["sinopsis"].str.strip() != ""].copy()
#Me quedo solo con los libros que tienen sinopsis y elimino los espacios de al principio y final de la sinopsis
#df_sinopsis

**Autores y genero de string a lista**

En el csv los autores y generos se guardaron como un string de json. Es una lista guardada como string, se decidio convertirla a una lista de Python para poder visualizar cuantos libros comparten géneros simultáneamente, ya que todos de ante mano pertenecen al género ciencia-ficción que fue elegido en la parte 1 del tp. 

In [5]:
df_sinopsis["autores"] = df_sinopsis["autores"].apply(json.loads)
df_sinopsis["generos"] = df_sinopsis["generos"].apply(json.loads)

print(df_sinopsis["generos"].explode().value_counts())
print("Géneros por libro (promedio):", df_sinopsis["generos"].apply(len).mean())

generos
Ciencia ficción               150
Novela                        109
Relato                         24
Fantástico                     21
Aventuras                      10
Juvenil                         5
Terror                          4
Intriga                         4
Otros                           4
Infantil y juvenil              3
Histórico                       2
Crítica y teoría literaria      2
Ensayo                          2
Publicaciónes periódicas        2
Romántico                       2
Infantil                        2
Bélico                          1
Poesía                          1
Cuentos                         1
Humor                           1
Ucronía                         1
Erótico                         1
Distopía                        1
Filosófico                      1
Name: count, dtype: int64
Géneros por libro (promedio): 2.36


**Normalización y limpieza**

*Dos versiones de texto: Uno crudo y otro limpio*

Texto crudo: sinopsis_emb
Texto limpio: sinopsis_limpio

In [6]:
df_sinopsis["sinopsis_emb"] = df_sinopsis["sinopsis"].apply(
    lambda x: re.sub(r"\s+", " ", x).strip()
)

#Normalizo los espacios en blanco de la sinopsis, reemplazando multiples espacios por uno solo y eliminando los espacios al principio y final de la sinopsis

In [7]:
# Carga las palabras stopwords en español
stop_words_es = set(stopwords.words("spanish"))

In [8]:
def limpiar_texto(texto: str) -> str:
    t = texto.lower()  # Pasa a minúscula
    t = re.sub(
        r"[^\w\s]", " ", t
    )  # Quita la puntuación y caracteres especiales
    tokens = word_tokenize(t)  # Tokenización
    filtrados = [
        w
        for w in tokens
        if w not in stop_words_es and len(w) > 1 and not w.isdigit()
    ]  # Stopwords y palabras de un solo caracter y números
    return " ".join(filtrados)


df_sinopsis["sinopsis_limpio"] = df_sinopsis["sinopsis_emb"].apply(limpiar_texto)

Guarda la nueva información en un archivo csv 

In [9]:
# Definimos la ruta
ruta_guardado = "../data/libros_preprocesados.csv"

# Guardamos el DataFrame en formato CSV
df_sinopsis.to_csv(ruta_guardado, index=False, encoding="utf-8")

print(f"Dataset preprocesado guardado exitosamente en: {ruta_guardado}")

Dataset preprocesado guardado exitosamente en: ../data/libros_preprocesados.csv


Decisión de preprocesamiento: 

Se mantienen dos versiones de las sinopsis. sinopsis_limpio se utiliza para técnicas basadas en frecuencia como TF-IDF, donde la eliminación de minúsculas, puntuación y palabras vacías reduce ruido. sinopsis_emb conserva el texto natural, únicamente normaliza los espacios, ya que los modelos de word embeddings aprovechan el orden de las palabras y las relaciones entre ellas. Aplicar la misma limpieza que en TF-IDF podría eliminar información relevante para representar el significado del texto.

**Parte B - Comparación entre modelos de word embeddings**

In [10]:
# !pip install gensim

In [11]:
from gensim.models import Word2Vec, KeyedVectors
from nltk.tokenize import sent_tokenize
from collections import Counter

In [12]:
# Letras del español + algunas de catalán/gallego que aparecen en el catálogo
PATRON_PALABRA = re.compile(r"[a-záéíóúüñàèìòùçï]+")

def tokenizar_w2v(texto):
    """Devuelve una lista de oraciones. Cada una es una lista de tokens en minúscula.
    Conserva stopwords porque Word2Vec las usa como contexto."""
    oraciones = sent_tokenize(texto, language="spanish")
    return [PATRON_PALABRA.findall(o.lower()) for o in oraciones]

oraciones_corpus = [
    oracion
    for texto in df_sinopsis["sinopsis_emb"]
    for oracion in tokenizar_w2v(texto)
    if len(oracion) > 1
]

In [13]:
frecuencias = Counter(t for o in oraciones_corpus for t in o)
print(f"Oraciones: {len(oraciones_corpus)}")
print(f"Tokens totales: {sum(frecuencias.values())}")
print(f"Vocabulario distinto: {len(frecuencias)}")
print(f"Palabras que aparecen una sola vez: {sum(1 for c in frecuencias.values() if c == 1)}")

Oraciones: 795
Tokens totales: 19128
Vocabulario distinto: 5234
Palabras que aparecen una sola vez: 3436


In [14]:
candidatas = ["robot", "nave", "planeta", "tierra", "futuro", "guerra", "tiempo",
              "humanidad", "espacio", "máquina", "viaje", "alienígena", "mundo", "ciudad"]

print(pd.Series({p: frecuencias[p] for p in candidatas}).sort_values(ascending=False))

mundo         77
tierra        37
planeta       36
guerra        34
futuro        29
nave          27
tiempo        25
humanidad     22
ciudad        19
viaje         12
espacio       11
alienígena     7
máquina        6
robot          2
dtype: int64


Entrenamiento del modelo propio

In [15]:
os.makedirs("../models", exist_ok=True)

modelo_propio = Word2Vec(
    sentences=oraciones_corpus,
    vector_size=100,   # corpus chico: 300 dimensiones serían demasiados parámetros para tan pocos datos
    window=5,          # contexto de ±5 palabras: captura relaciones temáticas sin mezclar demasiado
    min_count=3,       # descarta palabras con <3 apariciones: su vector sería casi aleatorio
    sg=1,              # skip-gram: funciona mejor que CBOW con corpus chicos y palabras poco frecuentes
    negative=10,       # negative sampling con 10 palabras negativas por ejemplo
    epochs=50,         # muchas pasadas porque hay pocos datos (el default es 5)
    seed=42,
    workers=1,         # con 1 hilo el entrenamiento es reproducible
)

print("Palabras en el vocabulario del modelo:", len(modelo_propio.wv))
modelo_propio.save("../models/w2v_propio.model")

Palabras en el vocabulario del modelo: 988


In [16]:
# ¿Cuánto vocabulario y cobertura se conserva con cada min_count?
tokens_contenido = [t for o in oraciones_corpus for t in o if t not in stop_words_es]

filas = []
for mc in [1, 2, 3, 5, 10]:
    vocab_mc = {w for w, c in frecuencias.items() if c >= mc}
    cubiertos = sum(1 for t in tokens_contenido if t in vocab_mc)
    filas.append({
        "min_count": mc,
        "palabras en vocabulario": len(vocab_mc),
        "% de tokens de contenido cubiertos": f"{cubiertos / len(tokens_contenido):.1%}",
    })

display(pd.DataFrame(filas))
print(f"Largo medio de oración: {sum(len(o) for o in oraciones_corpus) / len(oraciones_corpus):.1f} tokens")

,min_count,palabras en vocabulario,% de tokens de contenido cubiertos
0,1,5234,100.0%
1,2,1798,66.5%
2,3,988,51.0%
3,5,470,34.5%
4,10,175,18.1%


Largo medio de oración: 24.1 tokens


**Parámetros del modelo propio**

El corpus tiene 795 oraciones y 19.128 tokens, con 5.234 palabras distintas. De ellas, 3.436 (el 65,6%) aparecen una sola vez. Es un corpus muy chico para Word2Vec y los parámetros se eligieron para compensar esa escasez.

La tabla anterior muestra que la cobertura cae muy rápido al subir `min_count`: el vocabulario del corpus está dominado por palabras de baja frecuencia, que es justamente lo que limita al modelo propio.

- **`sg=1` (skip-gram).** con un corpus donde la mayoría de las palabras son raras, skip-gram aprovecha mejor los pocos datos.

- **`min_count=3`.** como el 65,6% del vocabulario son palabras únicas, incluirlas metería mucho ruido. El costo es alto: el vocabulario baja de 5.234 a 988 palabras, que cubren solo el 51,0% de los tokens de contenido, y palabras centrales del dominio como "robot" (2 apariciones) quedan afuera. Con `min_count=2` se ganarían 810 palabras y la cobertura subiría al 66,5%, pero esas palabras tendrían vectores entrenados a partir de solo dos contextos. Con `min_count=5`, en cambio, la cobertura caería al 34,5%, y el vector de cada sinopsis se armaría con apenas un tercio de sus palabras. El valor 3 es un punto intermedio entre la calidad de los vectores y la cobertura.

- **`window=5`.** Se consideran 5 palabras a cada lado. Con oraciones de 24,1 tokens en promedio, la ventana cubre casi media oración. Ventanas chicas (2 o 3) tienden a capturar similitud sintáctica (palabras que cumplen la misma función), y ventanas más grandes, similitud temática. Para un buscador de libros interesa lo temático, por eso no se redujo la ventana. Tampoco se agrandó, porque en oraciones de este largo una ventana mayor terminaría abarcando la oración entera.

- **`vector_size=100`.** Cada palabra tiene dos vectores entrenables (el de palabra y el de contexto), así que con 988 palabras y 100 dimensiones el modelo ajusta unos 200.000 parámetros a partir de 19 mil tokens. Con 300 dimensiones, como SBW, serían 600.000. Incluso 100 es mucho para este volumen de datos; se eligió por ser un valor habitual para corpus chicos, no por una optimización.

- **Otros parámetros.** `negative=10`: se usan 10 palabras negativas por ejemplo, un valor recomendado para corpus chicos. `epochs=50`: como hay pocos datos, se hacen muchas más pasadas que las 5 por defecto. `seed=42` y `workers=1`: con un solo hilo el entrenamiento es reproducible.

**Limitación.** Los parámetros no se optimizaron contra una métrica. Como muestran los vecinos de la sección siguiente, con 19 mil tokens el modelo propio queda limitado por la falta de datos más que por los parámetros elegidos.


**Modelo SBW-vectors-300-min5**

Se descargó el binario indicado como SBW-vectors-300-min5 desde la página oficial del SBWCE (mirror de Google Drive). El archivo recibido se llama sbwce.wordvectors.all.bin y tiene 1.692.025 palabras de dimensión 300, mientras que la página informa 1.000.653 para esa versión. Se utilizó este archivo. Las diferencias de vocabulario no afectan la comparación, que se hace sobre palabras frecuentes del corpus.

In [17]:
RUTA_SBW = "../models/sbwce.wordvectors.all.bin"
RUTA_SBW_KV = "../models/sbw_300.kv"

if os.path.exists(RUTA_SBW_KV):
    sbw = KeyedVectors.load(RUTA_SBW_KV, mmap="r")
else:
    # binary=True para el .bin(.gz); si tu archivo es .txt o .txt.bz2, usá binary=False
    sbw = KeyedVectors.load_word2vec_format(RUTA_SBW, binary=True)
    sbw.save(RUTA_SBW_KV)

print("Vocabulario SBW:", len(sbw), "| Dimensión:", sbw.vector_size)

# Verificamos cómo está normalizado SBW (mayúsculas, tildes) para tokenizar igual
for w in ["madrid", "Madrid", "alienígena", "alienigena", "robot"]:
    print(f"{w!r:15} -> {w in sbw.key_to_index}")

Vocabulario SBW: 1692025 | Dimensión: 300
'madrid'        -> True
'Madrid'        -> True
'alienígena'    -> True
'alienigena'    -> True
'robot'         -> True


In [18]:
def vecinos(kv, palabra, k=8, restrict_vocab=None):
    if palabra not in kv.key_to_index:
        return ["(fuera de vocabulario)"] + [""] * (k - 1)
    return [f"{w} ({s:.2f})" for w, s in kv.most_similar(palabra, topn=k, restrict_vocab=restrict_vocab)]

palabras_dominio = ["mundo", "planeta", "guerra", "futuro", "nave", "humanidad", "robot"]

for p in palabras_dominio:
    tabla = pd.DataFrame(
        {"Word2Vec propio": vecinos(modelo_propio.wv, p),
         # restrict_vocab: busca vecinos solo entre las 300.000 palabras más frecuentes de SBW,
         # para evitar errores de tipeo y palabras rarísimas como vecinos
         "SBW pre-entrenado": vecinos(sbw, p, restrict_vocab=300_000)},
        index=range(1, 9),
    )
    print(f"\n'{p}' (frecuencia en el corpus: {frecuencias[p]}) ===")
    display(tabla)


'mundo' (frecuencia en el corpus: 77) ===


,Word2Vec propio,SBW pre-entrenado
1,muro (0.53),país (0.67)
2,ambiente (0.51),hombre (0.60)
3,controlado (0.50),orbe (0.59)
4,lejos (0.50),todo (0.58)
5,conocen (0.50),universo (0.57)
6,telepatía (0.50),porque (0.57)
7,simulador (0.49),esto (0.56)
8,anillo (0.49),civilizado (0.56)



'planeta' (frecuencia en el corpus: 36) ===


,Word2Vec propio,SBW pre-entrenado
1,rojo (0.61),Tierra (0.79)
2,destruir (0.57),planetas (0.71)
3,bestias (0.55),universo (0.68)
4,humo (0.55),Titán (0.68)
5,mañana (0.53),Marte (0.67)
6,sobrevivir (0.53),galaxia (0.66)
7,primeros (0.52),Alfa_Centauri (0.65)
8,pandora (0.52),Cinturón_de_Kuiper (0.65)



'guerra' (frecuencia en el corpus: 34) ===


,Word2Vec propio,SBW pre-entrenado
1,horus (0.63),Guerra (0.70)
2,nuclear (0.59),contienda (0.70)
3,zombis (0.59),batalla (0.64)
4,futura (0.56),guerras (0.63)
5,extinción (0.54),fratricida (0.63)
6,poner (0.54),fraticida (0.62)
7,borde (0.54),conflagración (0.61)
8,extres (0.53),intestina (0.60)


... (4 outputs left out)


'humanidad' (frecuencia en el corpus: 22) ===


,Word2Vec propio,SBW pre-entrenado
1,acaba (0.73),Humanidad (0.82)
2,finalmente (0.68),civilización (0.62)
3,plaga (0.65),nación (0.58)
4,secreto (0.63),maldad (0.56)
5,fuerza (0.61),lesa (0.55)
6,recursos (0.61),barbarie (0.55)
7,extinción (0.60),humana (0.55)
8,superviviente (0.57),inmortalidad (0.54)



'robot' (frecuencia en el corpus: 2) ===


,Word2Vec propio,SBW pre-entrenado
1,(fuera de vocabulario),androide (0.77)
2,,robótico (0.76)
3,,cyborg (0.73)
4,,robots (0.70)
5,,humanoide (0.69)
6,,clon (0.67)
7,,monstruo (0.67)
8,,Hermano_Ojo (0.67)


**Resultados de los vecinos**

El modelo propio da similitudes de 0,49 a 0,73 y mezcla vecinos temáticos ("nave" → "hibernación", "bordo", "extraterrestre"; "guerra" → "nuclear", "extinción") con palabras que solo coinciden en alguna sinopsis puntual: nombres propios como "horus", "pandora" o "frank", y palabras como "durante", "tampoco" o "finalmente". "Robot" no tiene vector porque aparece solo 2 veces.

SBW devuelve vecinos coherentes ("planeta" → "Marte", "galaxia"; "guerra" → "contienda", "batalla"; "robot" → "androide", "cyborg"), pero con variantes de mayúscula ("Guerra", "Humanidad"), errores de tipeo ("fraticida", "fututo") y un sesgo de dominio: para "nave" devuelve "bóveda", "girola" y "ábside", es decir, la nave de una iglesia y no una nave espacial.

In [19]:
def vector_documento(texto, kv, quitar_stopwords=True):
    """Promedio de vectores de palabra, normalizado L2.
    Devuelve (vector o None, cobertura)."""
    tokens = [t for oracion in tokenizar_w2v(texto) for t in oracion]
    if quitar_stopwords:
        tokens = [t for t in tokens if t not in stop_words_es]

    def buscar_clave(t):
        if t in kv.key_to_index:
            return t
        if t.capitalize() in kv.key_to_index:   # SBW conserva mayúsculas: "marte" -> "Marte"
            return t.capitalize()
        return None

    presentes = [c for c in (buscar_clave(t) for t in tokens) if c is not None]
    cobertura = len(presentes) / len(tokens) if tokens else 0.0

    if not presentes:
        return None, cobertura

    v = np.mean([kv[c] for c in presentes], axis=0)
    norma = np.linalg.norm(v)
    if norma == 0 or not np.isfinite(v).all():
        return None, cobertura
    return (v / norma).astype(np.float32), cobertura


for nombre, kv in [("propio", modelo_propio.wv), ("sbw", sbw)]:
    resultados = df_sinopsis["sinopsis_emb"].apply(lambda t: vector_documento(t, kv))
    df_sinopsis[f"vec_{nombre}"] = resultados.str[0]
    df_sinopsis[f"cob_{nombre}"] = resultados.str[1]
    print(f"{nombre:7} | documentos sin vector: {df_sinopsis[f'vec_{nombre}'].isna().sum()}"
          f" | cobertura media: {df_sinopsis[f'cob_{nombre}'].mean():.1%}"
          f" | cobertura mínima: {df_sinopsis[f'cob_{nombre}'].min():.1%}")

# Guardamos en disco para no recalcular
df_sinopsis.to_pickle("../data/libros_vectores.pkl")

propio  | documentos sin vector: 0 | cobertura media: 50.9% | cobertura mínima: 19.5%
sbw     | documentos sin vector: 0 | cobertura media: 97.0% | cobertura mínima: 83.6%


**Vector de documento**

Cada sinopsis se representa con el promedio de los vectores de sus palabras, sin stopwords y normalizado. Las palabras sin vector se descartan, y para SBW se busca también la versión con mayúscula inicial ("marte" → "Marte").

Cobertura media: 50,9% en el modelo propio (mínima 19,5%) y 97,0% en SBW (mínima 83,6%). El vector del modelo propio se arma, en promedio, con la mitad de las palabras de cada sinopsis. Ningún documento quedó sin vector.

**Qué se pierde al promediar**

Además del orden: las negaciones ("no es una historia de guerra" ≈ "una historia de guerra"), quién hace qué ("los robots destruyen a la humanidad" = "la humanidad destruye a los robots"), el peso de las palabras clave en sinopsis largas, las palabras sin vector (por eso se informa la cobertura) y la distinción "Tierra"/"tierra" de SBW, por pasar todo a minúscula.

Se usa igual porque es barato, capta el tema general de cada sinopsis y sirve como punto de comparación con SBERT.

PARTE E

In [20]:
from dotenv import load_dotenv

# Carga las variables del archivo .env (está en la raíz del repo, un nivel arriba de src/)
encontrado = load_dotenv("../.env")
if not encontrado:
    raise FileNotFoundError("No se encontró el archivo .env en la raíz del repo.")

DATABASE_URL = os.environ.get("DATABASE_URL")
if not DATABASE_URL:
    raise RuntimeError("El archivo .env existe pero no tiene la variable DATABASE_URL.")

print("Credenciales cargadas desde .env")  # nunca imprimir la URL: tiene la contraseña

Credenciales cargadas desde .env


In [21]:
try:
    # Establecemos la conexión
    conn = psycopg2.connect(DATABASE_URL)
    cur = conn.cursor()

    # Nos aseguramos de que la extensión vectorial esté activa
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

    # Creamos la tabla para Word2Vec (Dimensión 300)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS libros_word2vec (
            id VARCHAR PRIMARY KEY,
            titulo VARCHAR,
            generos VARCHAR,
            embedding vector(300)
        );
    """)

    # Creamos la tabla para SBERT (Dimensión 512)
    cur.execute("""
        CREATE TABLE IF NOT EXISTS libros_sbert (
            id VARCHAR PRIMARY KEY,
            titulo VARCHAR,
            generos VARCHAR,
            embedding vector(512)
        );
    """)

    # Confirmamos los cambios en la base de datos
    conn.commit()
    print("¡Conexión exitosa! Las tablas 'libros_word2vec' y 'libros_sbert' fueron creadas.")

except Exception as e:
    print(f"Ocurrió un error: {e}")
finally:
    # Cierra la conexión al terminar
    if 'conn' in locals() and conn:
        cur.close()
        conn.close()

¡Conexión exitosa! Las tablas 'libros_word2vec' y 'libros_sbert' fueron creadas.
